# Hour 1 · Corpora and data


**Goal:** see how a clay tablet becomes a *programmatically accessible corpus* — not an e-book, but a graph of objects (tablet → line → word → sign) with features.

We use the **Copenhagen Ugaritic Corpus (CUC)** through line-level JSONL files hosted at HuggingFace (`AlexWalhai/cuc`). The original corpus is `DT-UCPH/cuc`. Licence: **CC BY-NC 4.0** (educational use).

*Reading:* `docs/02-corpora-and-data.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first ===
# Loads the Copenhagen Ugaritic Corpus (CUC) from the HuggingFace JSONL cache. No edits needed.
import sys
sys.path.append("..")                       # so Python can find data/loader.py
from data.loader import load_texts

texts = load_texts()                        # 278 real KTU tablets; cached after first download
print(f"Loaded {len(texts)} tablets.")
texts[0]["ktu"], texts[0]["genre"], texts[0]["lines"][:2]

## 1. One tablet, two scripts
Every line carries both a Latin transliteration (`lines`) and the actual **cuneiform** (`ugaritic`), plus a reference (`refs`).

In [ ]:
t = [x for x in texts if x["ktu"] == "1.3"][0]   # Baal Cycle
for ref, lat, cun in zip(t["refs"][:6], t["lines"][:6], t["ugaritic"][:6]):
    print(f'{ref:14s} {lat:28s} {cun}')

## 2. The corpus is a graph of object types
In Text-Fabric the CUC objects are **tablet · column · line · word · sign**. Our loader keeps tablets, lines, word tokens, and signs.

In [ ]:
print("tablets :", len(texts))
print("lines   :", sum(len(t["lines"]) for t in texts))
print("words   :", sum(len(t["tokens"]) for t in texts))
from data.loader import sign_counts
print("signs   :", sum(sign_counts(texts).values()))

## 3. Browse by genre
Genre labels are heuristic (KTU number + known tablets).

In [ ]:
from data.loader import texts_by_genre
for g, items in sorted(texts_by_genre(texts).items(), key=lambda kv: -len(kv[1])):
    print(f'{g:20s} {len(items):3d}  e.g. {", ".join(x["ktu"] for x in items[:5])}')

## 4. Simple queries
A corpus lets us *ask questions* in code.

In [ ]:
# (a) find every tablet that contains a given word form
TARGET = "mlk"   # "king" — TODO try "bʿl", "ilm", "ydd"
hits = [t["ktu"] for t in texts if TARGET in t["tokens"]]
print(f'{TARGET!r} appears in {len(hits)} tablets:', hits[:15], "...")

In [ ]:
# (b) print every line of one tablet that contains the word
ktu = "1.4"
one = [t for t in texts if t["ktu"] == ktu][0]
for ref, line in zip(one["refs"], one["lines"]):
    if TARGET in line.replace(".", " ").split():
        print(f'{ref}: {line}')

In [ ]:
# (c) how long is each letter?
letters = [t for t in texts if t["genre"] == "letter"]
for t in sorted(letters, key=lambda x: -len(x["tokens"]))[:8]:
    print(f'{t["ktu"]:7s} {len(t["tokens"]):4d} words')

## 5. Discussion / next step
A corpus is a **graph of objects and features**, queryable in code. CUC shares the Text-Fabric model with **BHSA** (Hebrew Bible) and **DSS** — the same queries port across corpora.

> **With full Text-Fabric** (`pip install text-fabric`; `use('DT-UCPH/cuc')`) you also get sign-level features: emendation, certainty, alternative readings.